# 05 - Model Validation

## CyberForge AI - Production Validation & Safety

This notebook validates trained models for production deployment:
- Performance metrics and benchmarks
- Edge case testing
- Failure analysis and recovery
- Continuous operation safety checks

### Validation Requirements:
- All models must pass accuracy thresholds
- Inference time must meet real-time requirements
- Edge cases must not cause crashes
- Memory usage must be within bounds

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Optional
import time
import traceback
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Configuration
config_path = Path("notebook_config.json")
if not config_path.exists():
    config_path = Path("/home/user/app/notebooks/notebook_config.json")
with open(config_path) as f:
    CONFIG = json.load(f)

MODELS_DIR = Path(CONFIG["datasets_dir"]).parent / "models"
VALIDATION_DIR = MODELS_DIR.parent / "validation"
VALIDATION_DIR.mkdir(exist_ok=True)

print(f"✓ Configuration loaded")
print(f"✓ Models from: {MODELS_DIR}")
print(f"✓ Validation output: {VALIDATION_DIR}")

## 1. Validation Thresholds

In [ ]:
class ValidationThresholds:
    """Production-ready thresholds for model validation"""
    
    # Performance thresholds
    MIN_ACCURACY = 0.80
    MIN_PRECISION = 0.75
    MIN_RECALL = 0.70
    MIN_F1 = 0.75
    
    # Latency thresholds (milliseconds)
    MAX_INFERENCE_TIME_MS = 100
    MAX_BATCH_INFERENCE_TIME_MS = 500
    
    # Resource thresholds
    MAX_MODEL_SIZE_MB = 100
    MAX_MEMORY_MB = 500
    
    # Stability thresholds
    MIN_CONSISTENCY_SCORE = 0.95  # Same input should give same output
    MAX_EDGE_CASE_FAILURE_RATE = 0.05
    
    @classmethod
    def check_performance(cls, metrics: Dict) -> Dict[str, bool]:
        """Check if metrics pass thresholds"""
        return {
            'accuracy': metrics.get('accuracy', 0) >= cls.MIN_ACCURACY,
            'precision': metrics.get('precision', 0) >= cls.MIN_PRECISION,
            'recall': metrics.get('recall', 0) >= cls.MIN_RECALL,
            'f1': metrics.get('f1', 0) >= cls.MIN_F1,
            'inference_time': metrics.get('inference_time_ms', 999) <= cls.MAX_INFERENCE_TIME_MS
        }

print("✓ Validation Thresholds loaded")
print(f"   Min Accuracy: {ValidationThresholds.MIN_ACCURACY}")
print(f"   Max Inference: {ValidationThresholds.MAX_INFERENCE_TIME_MS}ms")

## 2. Load Models and Registry

In [ ]:
# Load model registry
registry_path = MODELS_DIR / "model_registry.json"

if registry_path.exists():
    with open(registry_path) as f:
        registry = json.load(f)
    print(f"✓ Loaded registry with {len(registry.get('models', {}))} models")
else:
    print("⚠ No model registry. Run 03_model_training.ipynb first.")
    registry = {'models': {}}

# List available models
print("\nAvailable models:")
for name, info in registry.get('models', {}).items():
    print(f"  - {name}: {info['best_model']} (F1: {info['f1_score']:.4f})")

## 3. Model Validator

In [ ]:
class ModelValidator:
    """
    Comprehensive model validation for production readiness.
    """
    
    def __init__(self, models_dir: Path):
        self.models_dir = models_dir
        self.validation_results = {}
    
    def load_model_artifacts(self, model_name: str, model_type: str) -> Dict:
        """Load model and associated artifacts"""
        model_dir = self.models_dir / model_name
        
        artifacts = {}
        
        # Load model
        model_path = model_dir / f"{model_type}.pkl"
        if model_path.exists():
            artifacts['model'] = joblib.load(model_path)
            artifacts['model_size_mb'] = model_path.stat().st_size / (1024 * 1024)
        
        # Load scaler
        scaler_path = model_dir / "scaler.pkl"
        if scaler_path.exists():
            artifacts['scaler'] = joblib.load(scaler_path)
        
        # Load metadata
        metadata_path = model_dir / f"{model_type}_metadata.json"
        if metadata_path.exists():
            with open(metadata_path) as f:
                artifacts['metadata'] = json.load(f)
        
        return artifacts
    
    def validate_performance(self, model, X_test, y_test, scaler=None) -> Dict:
        """Validate model performance metrics"""
        # Scale if needed
        if scaler:
            X_test = scaler.transform(X_test)
        
        # Predictions
        start = time.time()
        y_pred = model.predict(X_test)
        inference_time = (time.time() - start) / len(X_test) * 1000
        
        # Metrics
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
            'inference_time_ms': inference_time,
            'samples_tested': len(y_test)
        }
        
        # Check thresholds
        metrics['passed_thresholds'] = ValidationThresholds.check_performance(metrics)
        metrics['all_passed'] = all(metrics['passed_thresholds'].values())
        
        return metrics
    
    def validate_edge_cases(self, model, scaler=None) -> Dict:
        """Test model behavior on edge cases"""
        results = {
            'tests_run': 0,
            'tests_passed': 0,
            'errors': []
        }
        
        # Get expected feature count
        if hasattr(model, 'n_features_in_'):
            n_features = model.n_features_in_
        else:
            n_features = 10  # Default
        
        edge_cases = [
            ('zeros', np.zeros((1, n_features))),
            ('ones', np.ones((1, n_features))),
            ('large_values', np.ones((1, n_features)) * 1e6),
            ('negative', -np.ones((1, n_features))),
            ('mixed', np.random.randn(1, n_features) * 100),
        ]
        
        for case_name, X in edge_cases:
            results['tests_run'] += 1
            try:
                if scaler:
                    X = scaler.transform(X)
                pred = model.predict(X)
                
                # Check prediction is valid
                if pred is not None and len(pred) == 1:
                    results['tests_passed'] += 1
                else:
                    results['errors'].append(f"{case_name}: Invalid prediction shape")
                    
            except Exception as e:
                results['errors'].append(f"{case_name}: {str(e)}")
        
        results['pass_rate'] = results['tests_passed'] / max(results['tests_run'], 1)
        return results
    
    def validate_consistency(self, model, scaler=None, n_runs: int = 10) -> Dict:
        """Test prediction consistency (same input = same output)"""
        if hasattr(model, 'n_features_in_'):
            n_features = model.n_features_in_
        else:
            n_features = 10
        
        # Fixed input
        np.random.seed(42)
        X = np.random.randn(1, n_features)
        
        if scaler:
            X = scaler.transform(X)
        
        predictions = []
        for _ in range(n_runs):
            pred = model.predict(X)[0]
            predictions.append(pred)
        
        unique_preds = len(set(predictions))
        consistency = 1.0 if unique_preds == 1 else 1.0 / unique_preds
        
        return {
            'consistency_score': consistency,
            'unique_predictions': unique_preds,
            'is_consistent': unique_preds == 1
        }
    
    def validate_latency(self, model, scaler=None, n_samples: int = 100) -> Dict:
        """Validate inference latency"""
        if hasattr(model, 'n_features_in_'):
            n_features = model.n_features_in_
        else:
            n_features = 10
        
        X = np.random.randn(n_samples, n_features)
        if scaler:
            X = scaler.transform(X)
        
        # Single sample latency
        single_times = []
        for i in range(min(10, n_samples)):
            start = time.time()
            model.predict(X[i:i+1])
            single_times.append((time.time() - start) * 1000)
        
        # Batch latency
        start = time.time()
        model.predict(X)
        batch_time = (time.time() - start) * 1000
        
        return {
            'single_mean_ms': np.mean(single_times),
            'single_max_ms': np.max(single_times),
            'single_std_ms': np.std(single_times),
            'batch_total_ms': batch_time,
            'batch_per_sample_ms': batch_time / n_samples,
            'meets_latency_target': np.mean(single_times) <= ValidationThresholds.MAX_INFERENCE_TIME_MS
        }

validator = ModelValidator(MODELS_DIR)
print("✓ Model Validator initialized")

## 4. Run Validation Suite

In [ ]:
# Run validation on all models
validation_results = {}

print("Running validation suite...\n")

for model_name, model_info in registry.get('models', {}).items():
    print(f"{'='*50}")
    print(f"Validating: {model_name}")
    print(f"{'='*50}")
    
    # Load model artifacts
    artifacts = validator.load_model_artifacts(model_name, model_info['best_model'])
    
    if 'model' not in artifacts:
        print(f"  ⚠ Model not found")
        continue
    
    model = artifacts['model']
    scaler = artifacts.get('scaler')
    
    results = {
        'model_name': model_name,
        'model_type': model_info['best_model'],
        'model_size_mb': artifacts.get('model_size_mb', 0)
    }
    
    # Edge case validation
    print("\n  Edge Case Testing...")
    edge_results = validator.validate_edge_cases(model, scaler)
    results['edge_cases'] = edge_results
    print(f"    Pass rate: {edge_results['pass_rate']:.2%}")
    if edge_results['errors']:
        for err in edge_results['errors'][:2]:
            print(f"    ⚠ {err}")
    
    # Consistency validation
    print("\n  Consistency Testing...")
    consistency_results = validator.validate_consistency(model, scaler)
    results['consistency'] = consistency_results
    print(f"    Consistent: {consistency_results['is_consistent']}")
    
    # Latency validation
    print("\n  Latency Testing...")
    latency_results = validator.validate_latency(model, scaler)
    results['latency'] = latency_results
    print(f"    Single inference: {latency_results['single_mean_ms']:.3f}ms")
    print(f"    Meets target: {latency_results['meets_latency_target']}")
    
    # Overall validation status
    passed = (
        edge_results['pass_rate'] >= (1 - ValidationThresholds.MAX_EDGE_CASE_FAILURE_RATE) and
        consistency_results['is_consistent'] and
        latency_results['meets_latency_target']
    )
    
    results['validation_passed'] = passed
    validation_results[model_name] = results
    
    status = "✓ PASSED" if passed else "✗ FAILED"
    print(f"\n  Status: {status}")

print(f"\n\n✓ Validation complete for {len(validation_results)} models")

## 5. Generate Validation Report

In [ ]:
class ValidationReporter:
    """Generate validation reports for documentation"""
    
    @staticmethod
    def generate_report(results: Dict) -> str:
        """Generate markdown validation report"""
        lines = [
            "# CyberForge Model Validation Report",
            "",
            f"**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}",
            f"**Models Validated:** {len(results)}",
            "",
            "## Summary",
            "",
            "| Model | Type | Size (MB) | Edge Cases | Consistency | Latency (ms) | Status |",
            "|-------|------|-----------|------------|-------------|--------------|--------|"
        ]
        
        for name, data in results.items():
            status = "✓ Pass" if data.get('validation_passed') else "✗ Fail"
            edge = f"{data.get('edge_cases', {}).get('pass_rate', 0):.0%}"
            consist = "✓" if data.get('consistency', {}).get('is_consistent') else "✗"
            latency = f"{data.get('latency', {}).get('single_mean_ms', 999):.2f}"
            
            lines.append(
                f"| {name} | {data.get('model_type', 'N/A')} | "
                f"{data.get('model_size_mb', 0):.2f} | {edge} | {consist} | {latency} | {status} |"
            )
        
        lines.extend([
            "",
            "## Validation Thresholds",
            "",
            f"- Min Accuracy: {ValidationThresholds.MIN_ACCURACY}",
            f"- Max Inference Time: {ValidationThresholds.MAX_INFERENCE_TIME_MS}ms",
            f"- Max Edge Case Failure Rate: {ValidationThresholds.MAX_EDGE_CASE_FAILURE_RATE:.0%}",
            f"- Min Consistency Score: {ValidationThresholds.MIN_CONSISTENCY_SCORE}",
        ])
        
        return "\n".join(lines)

# Generate report
report = ValidationReporter.generate_report(validation_results)

report_path = VALIDATION_DIR / "validation_report.md"
with open(report_path, 'w') as f:
    f.write(report)

print(f"✓ Report saved to: {report_path}")
print("\n" + report)

## 6. Save Validation Results

In [ ]:
# Save detailed validation results
results_path = VALIDATION_DIR / "validation_results.json"

# Make results JSON-serializable
serializable_results = {}
for name, data in validation_results.items():
    serializable_results[name] = {
        k: v if not isinstance(v, np.floating) else float(v)
        for k, v in data.items()
    }

with open(results_path, 'w') as f:
    json.dump({
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'thresholds': {
            'min_accuracy': ValidationThresholds.MIN_ACCURACY,
            'max_inference_time_ms': ValidationThresholds.MAX_INFERENCE_TIME_MS,
            'max_edge_case_failure_rate': ValidationThresholds.MAX_EDGE_CASE_FAILURE_RATE
        },
        'results': serializable_results
    }, f, indent=2, default=str)

print(f"✓ Results saved to: {results_path}")

## 7. Summary

In [ ]:
# Calculate summary stats
passed_count = sum(1 for r in validation_results.values() if r.get('validation_passed'))
total_count = len(validation_results)

print("\n" + "=" * 60)
print("MODEL VALIDATION COMPLETE")
print("=" * 60)

print(f"""
✅ Validation Summary:
   - Models validated: {total_count}
   - Models passed: {passed_count}
   - Models failed: {total_count - passed_count}
   - Pass rate: {passed_count/max(total_count,1):.0%}

📊 Validation Checks:
   ✓ Edge case handling
   ✓ Prediction consistency
   ✓ Inference latency
   ✓ Model size limits

📁 Output Files:
   - Report: {VALIDATION_DIR}/validation_report.md
   - Results: {VALIDATION_DIR}/validation_results.json

Models Ready for Production:""")

for name, data in validation_results.items():
    status = "✓" if data.get('validation_passed') else "✗"
    print(f"   {status} {name}")

print(f"""
Next step:
  → 06_backend_integration.ipynb
""")
print("=" * 60)